# 02. Preprocessing & Feature Engineering

## 🎯 Objective
Prepare the demand data for ML modeling (Segments A/B) and Statistical Forecasting (Segments C/D/E).

### ❌ Previous Mistakes
1. **Zero-Filling**: Provided an artificial zero for every missing day, polluting the data with 80% zeros.
2. **Lag Features**: Used `lag_1d`, `rolling_mean_7d` which assume a continuous daily time series. This fails for sparse demand.
3. **Inefficiency**: Calculated features for ALL products, even those that don't needed ML.

### ✅ Correct Approach
1. **No Zero-Filling**: Keep only actual order events.
2. **Effective Features**: Use statistical features (`order_frequency`, `dow_avg_demand`) and temporal features (`month`, `day_of_week`).
3. **Segment First**: Classify products into A, B, C, D, E. Only generate ML features for A/B.

In [19]:
import pandas as pd
import numpy as np
import os

# Load daily demand (output from 01)
input_path = "../../data/raw/demand_daily.csv"
demand_df = pd.read_csv(input_path)
demand_df['date'] = pd.to_datetime(demand_df['date'])

print(f"Original data: {len(demand_df):,} rows")
print("Note: Only actual order days, NO filled zeros!")

Original data: 150,999 rows
Note: Only actual order days, NO filled zeros!


C:\Users\Afaf\AppData\Local\Temp\ipykernel_21424\1113169216.py:7: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  demand_df = pd.read_csv(input_path)


## 1. Feature Engineering (Temporal)

In [20]:
def add_temporal_features(df):
    df = df.copy()
    
    # Basic temporal
    df['day_of_week'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
    df['quarter'] = df['date'].dt.quarter
    
    # Binary flags
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_month_start'] = (df['day'] <= 7).astype(int)
    df['is_month_end'] = (df['day'] >= 24).astype(int)
    
    # Cyclical encoding (for ML models)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    
    # Holidays (Hardcoded)
    holidays = [(1, 1), (12, 25), (5, 1), (7, 14), (11, 1), (11, 11)]
    df['is_holiday'] = df['date'].apply(lambda x: 1 if (x.month, x.day) in holidays else 0)

    return df

demand_with_temporal = add_temporal_features(demand_df)

## 2. Product Statistics

In [21]:
def calculate_product_stats(df):
    total_days_span = (df.date.max() - df.date.min()).days + 1
    
    # Group by product
    product_stats = df.groupby('id_produit').agg({
        'quantite_demande': ['count', 'sum', 'mean', 'std'],
    }).reset_index()
    
    # Fix multi-index columns
    product_stats.columns = ['id_produit', 'order_count', 'total_demand', 'avg_demand', 'std_demand']
    
    # Metrics
    # Order frequency (orders per day over the full dataset range proxy)
    product_stats['order_frequency'] = product_stats['order_count'] / total_days_span
    
    # Avg Daily Volume (Total Demand / Total Days)
    product_stats['avg_daily_vol'] = product_stats['total_demand'] / total_days_span

    # Coefficient of variation (Stability)
    product_stats['cv'] = product_stats['std_demand'] / (product_stats['avg_demand'] + 1)
    
    return product_stats

product_stats = calculate_product_stats(demand_with_temporal)

# Merge stats back to main dataframe
demand_enriched = demand_with_temporal.merge(
    product_stats[['id_produit', 'order_frequency', 'cv', 'avg_demand', 'avg_daily_vol']], 
    on='id_produit'
)

## 3. Advanced Feature: Day-of-Week Averages
Captures usage patterns (e.g., "Does this product sell more on Fridays?")

In [22]:
dow_stats = demand_enriched.groupby(['id_produit', 'day_of_week'])['quantite_demande'].mean().reset_index()
dow_stats.columns = ['id_produit', 'day_of_week', 'dow_avg_demand']

demand_enriched = demand_enriched.merge(dow_stats, on=['id_produit', 'day_of_week'], how='left')

## 4. Segmentation

In [23]:
def segment_products(stats_df):
    def classify(row):
        freq = row['order_frequency']
        vol = row['avg_daily_vol']
        
        if freq > 0.5: # Ordered every other day or more
            if vol > 50:
                return 'A_HIGH_FREQ_HIGH_VOL'  # ML
            else:
                return 'B_HIGH_FREQ_LOW_VOL'   # ML
        elif freq > 0.1: # At least once every 10 days
            return 'C_MEDIUM_FREQ'  # WMA
        else:
            return 'D_LOW_FREQ'  # Naive
    
    stats_df['segment'] = stats_df.apply(classify, axis=1)
    return stats_df

product_stats = segment_products(product_stats)
print("Product segmentation:")
print(product_stats['segment'].value_counts())

# Save segments
if not os.path.exists("../../data/processed"):
    os.makedirs("../../data/processed")

product_stats.to_csv("../../data/processed/product_segments.csv", index=False)

# Merge segment info into main DF
demand_enriched = demand_enriched.merge(product_stats[['id_produit', 'segment']], on='id_produit')

Product segmentation:
segment
D_LOW_FREQ              834
C_MEDIUM_FREQ           184
A_HIGH_FREQ_HIGH_VOL    102
B_HIGH_FREQ_LOW_VOL       9
Name: count, dtype: int64


## 5. Prepare ML Data (Segments A & B only)

In [24]:
ml_segments = ['A_HIGH_FREQ_HIGH_VOL', 'B_HIGH_FREQ_LOW_VOL']
ml_data = demand_enriched[demand_enriched['segment'].isin(ml_segments)].copy()

print(f"\nML data (Segments A & B): {len(ml_data):,} rows")

# One-hot encode category for ML
ml_data = pd.get_dummies(ml_data, columns=['categorie'], prefix='cat')

# Save Feature Columns for later use
feature_cols_candidates = [
    'day_of_week', 'month', 'week_of_year', 'quarter',
    'is_weekend', 'is_month_start', 'is_month_end', 'is_holiday',
    'month_sin', 'month_cos', 'dow_sin', 'dow_cos',
    'colisage fardeau', 'volume pcs (m3)', 'Is_Gerbable',
    'order_frequency', 'cv', 'avg_demand', 'dow_avg_demand'
] + [col for col in ml_data.columns if col.startswith('cat_')]

# Save ML dataset
ml_data.to_csv("../../data/processed/train_ml_full.csv", index=False)

# Temporal Split for Training/Testing
split_date = pd.to_datetime('2026-01-01')

train = ml_data[ml_data['date'] < split_date]
test = ml_data[ml_data['date'] >= split_date]

train.to_csv("../../data/processed/train_ml.csv", index=False)
test.to_csv("../../data/processed/test_ml.csv", index=False)

print(f"\nTrain: {len(train):,} rows ({train['date'].min().date()} to {train['date'].max().date()})")
print(f"Test: {len(test):,} rows ({test['date'].min().date()} to {test['date'].max().date()})")
print("\n✓ Preprocessing complete!")


ML data (Segments A & B): 107,881 rows

Train: 106,283 rows (2024-01-02 to 2025-12-30)
Test: 1,598 rows (2026-01-03 to 2026-01-08)

✓ Preprocessing complete!
